In [ ]:
import os, re, glob
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd

# ===================== USER CONFIG =====================
TARGET_PLEV_HPA = 50.0          # <- change to 10/50/100/500 freely
PLEV_TARGET_PA  = TARGET_PLEV_HPA * 100.0

LEVEL_TAG = f"{int(TARGET_PLEV_HPA)}" if float(TARGET_PLEV_HPA).is_integer() else f"{TARGET_PLEV_HPA}".replace(".", "p")
VAR_TAG   = f"Z{LEVEL_TAG}"      # e.g., Z50

# Lead constraint (YOU SAID: MIN_LEAD = 1)
MIN_LEAD = 1

# ---------- paths ----------
HINDCAST_BASE_DIR = Path("/home/weiji/restart_exam/hindcast_data")  # case folders: 0008-02*, etc.
B2000_Z3_HYB_DIR  = "/home/weiji/restart_exam/longrun_B2000WCN_withchem_data/Z3"
BWCN_H3_DIR       = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"

# optional filter (None for all)
TARGET_YEARS = {"0003","0008","0013","0014","0019"}  # or None

# output root for monthly pipeline
BASE_OUT = Path(f"/home/weiji/restart_exam/code/2060202HINDCAST/result/{VAR_TAG}_monthly")
BASE_OUT.mkdir(parents=True, exist_ok=True)

OUT_CLIM12      = BASE_OUT / f"{VAR_TAG}_climatology_12mon.nc"
OUT_BWCN_ANOM   = BASE_OUT / f"{VAR_TAG}_bwcn_monthly_anom.nc"
OUT_HIND_MMON   = BASE_OUT / "hind_monthly_fields"
OUT_HIND_ANOM   = BASE_OUT / "hind_monthly_anom"
OUT_ACC_DIR     = BASE_OUT / "ACC_monthly"
OUT_FIG_DIR     = BASE_OUT / f"fig_monthly_grouped_{VAR_TAG}"

for d in [OUT_HIND_MMON, OUT_HIND_ANOM, OUT_ACC_DIR, OUT_FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUT_ACC_CSV = OUT_ACC_DIR / "ACC_monthly_summary.csv"

# ---------- IO strategy ----------
ENGINE   = "netcdf4"
PARALLEL = False
CHUNKS   = {"time": 30}   # safe for monthly resample

COMPRESS_LEVEL = 4
USE_FLOAT32 = True

# standard pressure levels (Pa)
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

print("CONFIG:")
print("  VAR_TAG =", VAR_TAG, "TARGET_PLEV_HPA =", TARGET_PLEV_HPA, "MIN_LEAD =", MIN_LEAD)
print("  BASE_OUT =", BASE_OUT)


In [ ]:
def parse_case(case_name: str) -> dict:
    m = re.match(r"^(?P<y>\d{4})-(?P<m>\d{2})(?P<suf>.*)$", case_name)
    if not m:
        raise ValueError(f"Bad case name: {case_name}")
    return {
        "case": case_name,
        "init_year": int(m.group("y")),
        "init_month": int(m.group("m")),
        "suffix": (m.group("suf") or "")
    }

def case_to_init_key(case: str) -> str:
    m = re.match(r"^(\d{4}-\d{2}).*$", case)
    if not m:
        return "UNKNOWN"
    return m.group(1)

def add_months(y: int, m: int, lead: int) -> tuple[int, int]:
    idx0 = (m - 1) + lead
    y2 = y + idx0 // 12
    m2 = (idx0 % 12) + 1
    return y2, m2

def infer_family_short(case: str) -> str:
    c = case.lower()
    if "v2" in c:
        return "LP"
    if "v3" in c:
        return "QP"
    return "SP"

def infer_ozone_mode(case: str) -> str:
    c = case.lower()
    if ("nocouple" in c) or ("nocoupl" in c):
        return "prescribed o3"
    return "interactive o3"

def coslat_weights(lat: xr.DataArray) -> np.ndarray:
    w = np.cos(np.deg2rad(lat.values))
    return np.maximum(w, 0.0).astype(np.float64)

def weighted_corr2d(a2d: np.ndarray, b2d: np.ndarray, w_lat: np.ndarray) -> float:
    if a2d.ndim != 2 or b2d.ndim != 2:
        raise ValueError("a2d/b2d must be 2D (lat,lon)")
    if a2d.shape != b2d.shape:
        raise ValueError("shape mismatch")

    lat_n, lon_n = a2d.shape
    w2d = np.repeat(w_lat.reshape(lat_n, 1), lon_n, axis=1)

    m = np.isfinite(a2d) & np.isfinite(b2d) & np.isfinite(w2d) & (w2d > 0)
    if m.sum() < 10:
        return np.nan

    aa, bb, ww = a2d[m], b2d[m], w2d[m]
    wsum = ww.sum()
    if wsum <= 0:
        return np.nan

    am = (ww * aa).sum() / wsum
    bm = (ww * bb).sum() / wsum
    a0 = aa - am
    b0 = bb - bm

    cov = (ww * a0 * b0).sum()
    va  = (ww * a0 * a0).sum()
    vb  = (ww * b0 * b0).sum()
    if va <= 0 or vb <= 0:
        return np.nan
    return float(cov / np.sqrt(va * vb))

def unify_vertical_dim(da: xr.DataArray) -> xr.DataArray:
    if ("plev_2" in da.dims) and ("plev" not in da.dims):
        da = da.rename({"plev_2": "plev"})
    if ("lev" in da.dims) and ("plev" not in da.dims):
        da = da.rename({"lev": "plev"})
    if ("level" in da.dims) and ("plev" not in da.dims):
        da = da.rename({"level": "plev"})
    if ("plev" in da.dims) and ("plev_2" in da.dims):
        da = da.isel(plev_2=0, drop=True)
    return da

def safe_open_mfdataset(files):
    return xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=PARALLEL,
        use_cftime=True,
        chunks=CHUNKS,
        engine=ENGINE,
    )

def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    hyam = ds["hyam"]
    hybm = ds["hybm"]
    P0   = ds["P0"]
    PS   = ds["PS"]
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def interp_profile_logp(v_hyb: xr.DataArray, p_hyb: xr.DataArray, p_tgt: np.ndarray) -> xr.DataArray:
    p_tgt = np.asarray(p_tgt, float)
    nplev = int(p_tgt.size)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full((nplev,), np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        idx = np.argsort(p_use)
        p_use = p_use[idx]
        v_use = v_use[idx]
        return np.interp(np.log(p_tgt), np.log(p_use), v_use, left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        output_sizes={"plev": nplev},
    )
    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out

def monthly_mean_then_interp_to_stdplev(ds: xr.Dataset, varname="Z3") -> xr.DataArray:
    need = [varname, "PS", "hyam", "hybm", "P0"]
    missing = [v for v in need if v not in ds.variables]
    if missing:
        raise RuntimeError(f"Dataset missing vars: {missing}")

    ds_m = ds[need].resample(time="MS").mean()
    v_m = ds_m[varname]
    p_m = compute_pressure_mid(ds_m)

    v_std = interp_profile_logp(v_m, p_m, PLEV_STD_PA)
    v_std = v_std.transpose("time", "plev", "lat", "lon")
    v_std.name = f"{varname}_stdplev"
    v_std.attrs["units"] = v_m.attrs.get("units", "")
    return v_std

def write_nc(ds: xr.Dataset, out_path: Path):
    enc = {}
    for v in ds.data_vars:
        enc[v] = {"zlib": True, "complevel": int(COMPRESS_LEVEL)}
        if np.issubdtype(ds[v].dtype, np.floating):
            enc[v].update({
                "dtype": "float32" if USE_FLOAT32 else str(ds[v].dtype),
                "_FillValue": np.float32(np.nan),
            })
    ds.to_netcdf(out_path, encoding=enc)


In [ ]:
if OUT_CLIM12.exists():
    print("[SKIP] exists:", OUT_CLIM12)
else:
    b2000_files = sorted(glob.glob(os.path.join(B2000_Z3_HYB_DIR, "*.Z3.hybrid.nc")))
    if len(b2000_files) == 0:
        raise RuntimeError(f"No B2000 hybrid Z3 files found: {B2000_Z3_HYB_DIR}")

    print("[B2000] open:", len(b2000_files), "files")
    ds_b2000 = safe_open_mfdataset(b2000_files)
    z3_std = monthly_mean_then_interp_to_stdplev(ds_b2000, "Z3")  # (time,plev,lat,lon)
    ds_b2000.close()

    zlev = z3_std.sel(plev=PLEV_TARGET_PA, method="nearest")  # (time,lat,lon)

    clim12 = zlev.groupby("time.month").mean("time").rename({"month": "month_of_year"})
    clim12["month_of_year"].attrs["long_name"] = "Month of year (1-12)"

    out_var = f"{VAR_TAG}_clim12"
    ds_out = xr.Dataset(
        data_vars={out_var: clim12.astype(np.float32 if USE_FLOAT32 else np.float64)},
        coords={"month_of_year": clim12["month_of_year"], "lat": clim12["lat"], "lon": clim12["lon"]},
        attrs={
            "source": "B2000WCN_withchem hybrid Z3 yearly files",
            "processing": f"monthly mean -> stdplev logp interp -> Z{TARGET_PLEV_HPA:g}hPa -> 12-mon climatology",
            "plev_target_hpa": float(TARGET_PLEV_HPA),
            "plev_target_pa": float(PLEV_TARGET_PA),
        }
    )
    if "units" in z3_std.attrs:
        ds_out[out_var].attrs["units"] = z3_std.attrs["units"]

    write_nc(ds_out, OUT_CLIM12)
    print("[SAVE]", OUT_CLIM12)


In [ ]:
if OUT_BWCN_ANOM.exists():
    print("[SKIP] exists:", OUT_BWCN_ANOM)
else:
    if not OUT_CLIM12.exists():
        raise FileNotFoundError("Run Cell 3 first: B2000 climatology missing")

    ds_clim = xr.open_dataset(OUT_CLIM12)
    clim12 = ds_clim[f"{VAR_TAG}_clim12"]  # (month_of_year,lat,lon)
    ds_clim.close()

    bwcn_files = sorted(glob.glob(os.path.join(BWCN_H3_DIR, "*.cam.h3.*.nc")))
    if len(bwcn_files) == 0:
        raise RuntimeError(f"No BWCN h3 files found: {BWCN_H3_DIR}")

    print("[BWCN] open:", len(bwcn_files), "files")
    ds_bwcn = safe_open_mfdataset(bwcn_files)
    z3_std = monthly_mean_then_interp_to_stdplev(ds_bwcn, "Z3")  # (time,plev,lat,lon)
    ds_bwcn.close()

    zlev = z3_std.sel(plev=PLEV_TARGET_PA, method="nearest")  # (time,lat,lon)

    month = zlev["time"].dt.month.astype(np.int16)
    clim_for_time = clim12.sel(month_of_year=month)  # broadcast along time
    anom_t = (zlev - clim_for_time)
    anom_t.name = f"{VAR_TAG}_bwcn_anom_time"

    # aggregate to (year,month)
    grouped = anom_t.groupby("time.year").map(lambda x: x.groupby("time.month").mean("time"))
    grouped = grouped.rename({"year": "year", "month": "month"})
    out_var = f"{VAR_TAG}_bwcn_anom_ym"
    grouped.name = out_var

    ds_out = xr.Dataset(
        data_vars={out_var: grouped.astype(np.float32 if USE_FLOAT32 else np.float64)},
        coords={"year": grouped["year"].astype(np.int32),
                "month": grouped["month"].astype(np.int16),
                "lat": grouped["lat"], "lon": grouped["lon"]},
        attrs={
            "note": "BWCN monthly anomalies wrt B2000 month-of-year climatology",
            "climatology_file": str(OUT_CLIM12),
            "plev_target_hpa": float(TARGET_PLEV_HPA),
        }
    )
    write_nc(ds_out, OUT_BWCN_ANOM)
    print("[SAVE]", OUT_BWCN_ANOM)


In [ ]:
def list_case_dirs(base_dir: Path):
    out = []
    for p in sorted(base_dir.iterdir()):
        if p.is_dir() and re.match(r"^\d{4}-\d{2}.*$", p.name):
            if TARGET_YEARS is not None:
                y4 = p.name.split("-")[0]
                if y4 not in TARGET_YEARS:
                    continue
            out.append(p)
    return out

def list_member_files(case_dir: Path, var_dirname="Z3", var_name="Z3"):
    vdir = case_dir / var_dirname
    if not vdir.is_dir():
        return []
    return sorted(vdir.glob(f"*.{var_name}.nc"))

def parse_member_id_from_filename(f: Path, case_name: str, fallback_idx: int) -> int:
    parts = f.name.split(".")
    if case_name in parts:
        i = parts.index(case_name)
        if i + 1 < len(parts) and parts[i + 1].isdigit():
            try:
                return int(parts[i + 1])
            except Exception:
                pass
    return fallback_idx

def open_member_da(f: Path, var_name="Z3"):
    ds = xr.open_dataset(f, decode_times=True, engine=ENGINE, chunks=CHUNKS)
    if var_name not in ds:
        ds.close()
        return None
    da = unify_vertical_dim(ds[var_name])
    ds.close()
    return da

case_dirs = list_case_dirs(HINDCAST_BASE_DIR)
print("[INFO] hind cases:", len(case_dirs))

for case_dir in case_dirs:
    case = case_dir.name
    out_path = OUT_HIND_MMON / f"{VAR_TAG}_hind_monthly_fields_{case}.nc"
    if out_path.exists():
        continue

    meta = parse_case(case)
    files = list_member_files(case_dir, var_dirname="Z3", var_name="Z3")
    if not files:
        print("[SKIP] no member files:", case)
        continue

    da_list = []
    for idx, f in enumerate(files, start=1):
        da = open_member_da(f, "Z3")
        if da is None:
            continue
        if "plev" not in da.dims:
            continue
        mid = parse_member_id_from_filename(f, case, idx)
        da = da.expand_dims("member").assign_coords(member=[mid])
        da_list.append(da)

    if not da_list:
        print("[SKIP] no valid members:", case)
        continue

    z_all = xr.concat(da_list, dim="member", join="outer").sortby("member")
    zlev = z_all.sel(plev=PLEV_TARGET_PA, method="nearest")  # (member,time,lat,lon)

    # monthly mean
    zmon = zlev.resample(time="MS").mean()   # (member,time,lat,lon)

    nlead = zmon.sizes["time"]
    leads = np.arange(nlead, dtype=np.int16)

    vy = np.zeros((nlead,), dtype=np.int32)
    vm = np.zeros((nlead,), dtype=np.int16)
    for i, ld in enumerate(leads):
        y2, m2 = add_months(meta["init_year"], meta["init_month"], int(ld))
        vy[i] = y2
        vm[i] = m2

    z_ld = zmon.rename({"time": "lead"}).assign_coords(lead=leads)
    z_ld = z_ld.assign_coords(verify_year=("lead", vy), verify_month=("lead", vm))

    out_var = f"{VAR_TAG}_hind_mmean"
    ds_out = xr.Dataset(
        data_vars={out_var: z_ld.astype(np.float32 if USE_FLOAT32 else np.float64)},
        coords={
            "member": z_ld["member"],
            "lead": z_ld["lead"],
            "lat": z_ld["lat"],
            "lon": z_ld["lon"],
            "verify_year": z_ld["verify_year"],
            "verify_month": z_ld["verify_month"],
        },
        attrs={
            "case_name": case,
            "init_year": int(meta["init_year"]),
            "init_month": int(meta["init_month"]),
            "suffix": meta["suffix"],
            "plev_target_hpa": float(TARGET_PLEV_HPA),
            "note": "Hindcast monthly mean at target plev; lead=0 corresponds to init month (by definition here).",
        }
    )

    write_nc(ds_out, out_path)
    print("[SAVE]", out_path)

print("[DONE] hind monthly fields ->", OUT_HIND_MMON)


In [ ]:
if not OUT_CLIM12.exists():
    raise FileNotFoundError("Need B2000 clim12; run Cell 3")
if not OUT_BWCN_ANOM.exists():
    raise FileNotFoundError("Need BWCN anom; run Cell 4")

# load clim + ref
ds_clim = xr.open_dataset(OUT_CLIM12)
clim12 = ds_clim[f"{VAR_TAG}_clim12"]  # (month_of_year,lat,lon)
lat = clim12["lat"]
lon = clim12["lon"]
w_lat = coslat_weights(lat)
ds_clim.close()

ds_bw = xr.open_dataset(OUT_BWCN_ANOM)
bw_anom = ds_bw[f"{VAR_TAG}_bwcn_anom_ym"]  # (year,month,lat,lon)
ds_bw.close()

hind_files = sorted(OUT_HIND_MMON.glob(f"{VAR_TAG}_hind_monthly_fields_*.nc"))
print("[INFO] hind monthly files:", len(hind_files))

rows = []

for f in hind_files:
    case = f.name.replace(f"{VAR_TAG}_hind_monthly_fields_", "").replace(".nc", "")
    ds_h = xr.open_dataset(f)

    var = f"{VAR_TAG}_hind_mmean"
    if var not in ds_h:
        ds_h.close()
        continue

    z = ds_h[var].reindex(lat=lat, lon=lon)   # (member,lead,lat,lon)

    leads = z["lead"].values.astype(int)
    vy = ds_h["verify_year"].values.astype(int)
    vm = ds_h["verify_month"].values.astype(int)

    # hind anomaly wrt B2000 month-of-year climatology (verify month)
    clim_for_lead = clim12.sel(month_of_year=xr.DataArray(vm, dims=("lead",)))
    hind_anom = (z - clim_for_lead)
    hind_anom.name = f"{VAR_TAG}_hind_anom"

    # save hind anomaly per case
    out_anom = OUT_HIND_ANOM / f"{VAR_TAG}_hind_monthly_anom_{case}.nc"
    if not out_anom.exists():
        ds_out = xr.Dataset(
            data_vars={hind_anom.name: hind_anom.astype(np.float32 if USE_FLOAT32 else np.float64)},
            coords={
                "member": hind_anom["member"],
                "lead": hind_anom["lead"],
                "lat": hind_anom["lat"],
                "lon": hind_anom["lon"],
                "verify_year": ("lead", vy.astype(np.int32)),
                "verify_month": ("lead", vm.astype(np.int16)),
            },
            attrs=dict(ds_h.attrs),
        )
        write_nc(ds_out, out_anom)
        print("[SAVE] hind anom:", out_anom)

    # compute ACC vs BWCN reference
    members = hind_anom["member"].values
    acc_member = np.full((members.size, leads.size), np.nan, dtype=np.float32 if USE_FLOAT32 else np.float64)
    acc_ens    = np.full((leads.size,), np.nan, dtype=np.float32 if USE_FLOAT32 else np.float64)

    for j, ld in enumerate(leads):
        if ld < MIN_LEAD:
            continue
        yv, mv = int(vy[j]), int(vm[j])
        if (yv not in bw_anom["year"].values) or (mv not in bw_anom["month"].values):
            continue

        ref2d = bw_anom.sel(year=yv, month=mv).values
        if not np.isfinite(ref2d).any():
            continue

        for i, mid in enumerate(members):
            a2d = hind_anom.sel(member=mid, lead=ld).values
            acc_member[i, j] = weighted_corr2d(a2d, ref2d, w_lat)

        ens2d = hind_anom.sel(lead=ld).mean("member").values
        acc_ens[j] = weighted_corr2d(ens2d, ref2d, w_lat)

        init_year = int(ds_h.attrs.get("init_year", -999))
        init_month = int(ds_h.attrs.get("init_month", -999))
        init_key = f"{init_year:04d}-{init_month:02d}"

        rows.append({
            "case": case,
            "init_key": init_key,
            "lead": int(ld),
            "verify_year": yv,
            "verify_month": mv,
            "verify_key": f"{yv:04d}-{mv:02d}",
            "ACC_ensmean": float(acc_ens[j]) if np.isfinite(acc_ens[j]) else np.nan,
            "family_short": infer_family_short(case),
            "ozone_mode": infer_ozone_mode(case),
            "target_hpa": float(TARGET_PLEV_HPA),
        })

    out_acc = OUT_ACC_DIR / f"ACC_monthly_{case}.nc"
    ds_acc = xr.Dataset(
        data_vars={
            "ACC_member": (("member","lead"), acc_member),
            "ACC_ensmean": (("lead",), acc_ens),
        },
        coords={
            "member": members,
            "lead": leads.astype(np.int16),
            "verify_year": ("lead", vy.astype(np.int32)),
            "verify_month": ("lead", vm.astype(np.int16)),
        },
        attrs={"case_name": case, "MIN_LEAD": int(MIN_LEAD)}
    )
    write_nc(ds_acc, out_acc)

    ds_h.close()

df_acc = pd.DataFrame(rows)
df_acc = df_acc.sort_values(["init_key","lead","verify_key","case"])
df_acc.to_csv(OUT_ACC_CSV, index=False)
print("[SAVE] ACC CSV:", OUT_ACC_CSV)
df_acc.head(20)


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.path as mpath

# -------- plotting config --------
MAP_PROJ = ccrs.NorthPolarStereo(central_longitude=0)
DATA_CRS = ccrs.PlateCarree()
POLAR_EXTENT = [0, 360, 20, 90]

CMAP = "RdBu_r"
COASTLINE_LW = 0.5
BORDER_LW = 0.3

FIGSIZE = (12.5, 8.5)
FS_TITLE = 10
FS_LABEL = 9
FS_TICK  = 8
VLIM_Q = 0.98

MAX_MAPS_PER_ROW = 3
BAR_HEIGHT = 0.60
BAR_Y_SPACING = 1.60
BAR_ROW_BASE = 0.75
BAR_ROW_PER_BAR = 0.060

def set_circular_boundary(ax):
    theta = np.linspace(0, 2*np.pi, 400)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * radius + center)
    ax.set_boundary(circle, transform=ax.transAxes)

def compute_common_vlim(fields_2d, q=0.98):
    arrs = []
    for f in fields_2d:
        v = f.values
        v = v[np.isfinite(v)]
        if v.size:
            arrs.append(np.abs(v))
    if not arrs:
        return 1.0
    allabs = np.concatenate(arrs)
    vmax = float(np.quantile(allabs, q))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(allabs)) if np.isfinite(allabs).any() else 1.0
    return vmax

def plot_polar_map(ax, field2d, title, vlim):
    ax.set_title(title, fontsize=FS_TITLE, pad=3)
    ax.set_extent(POLAR_EXTENT, crs=DATA_CRS)
    ax.coastlines(linewidth=COASTLINE_LW)
    ax.add_feature(cfeature.BORDERS, linewidth=BORDER_LW, alpha=0.7)
    set_circular_boundary(ax)
    im = ax.pcolormesh(field2d["lon"], field2d["lat"], field2d,
                       transform=DATA_CRS, cmap=CMAP,
                       vmin=-vlim, vmax=vlim, shading="auto")
    return im

# ---- load acc table (for bars) ----
if not OUT_ACC_CSV.exists():
    raise FileNotFoundError("ACC CSV missing, run Cell 6 first.")
df_acc = pd.read_csv(OUT_ACC_CSV)
df_acc["case"] = df_acc["case"].astype(str)
df_acc["verify_key"] = df_acc["verify_key"].astype(str)

def lookup_acc(case: str, verify_key: str, lead: int) -> float:
    sub = df_acc[(df_acc["case"] == case) & (df_acc["verify_key"] == verify_key) & (df_acc["lead"] == lead)]
    if sub.empty:
        return np.nan
    v = sub["ACC_ensmean"].iloc[0]
    return float(v) if np.isfinite(v) else np.nan

# ---- load BWCN ref ----
ds_bw = xr.open_dataset(OUT_BWCN_ANOM)
bw_anom = ds_bw[f"{VAR_TAG}_bwcn_anom_ym"]  # (year,month,lat,lon)
ds_bw.close()

# ---- scan hind anomaly files to define groups ----
hind_anom_files = sorted(OUT_HIND_ANOM.glob(f"{VAR_TAG}_hind_monthly_anom_*.nc"))
if len(hind_anom_files) == 0:
    raise RuntimeError(f"No hind anomaly files found in {OUT_HIND_ANOM}")

cases_all = [f.name.replace(f"{VAR_TAG}_hind_monthly_anom_", "").replace(".nc", "") for f in hind_anom_files]

# group by init_key=YYYY-MM (first 7 chars)
groups = {}
for case in cases_all:
    ik = case_to_init_key(case)
    groups.setdefault(ik, []).append(case)

def sort_case_key(case: str):
    ozone_rank = 0 if infer_ozone_mode(case) == "interactive o3" else 1
    fam = infer_family_short(case)
    return (ozone_rank, fam, case)

for ik in groups:
    groups[ik] = sorted(groups[ik], key=sort_case_key)

init_keys = sorted(groups.keys())
print("[INFO] init groups:", init_keys)

def get_verify_targets_from_case(case: str):
    """read lead->verify from file coords, avoid relying on CSV"""
    f = OUT_HIND_ANOM / f"{VAR_TAG}_hind_monthly_anom_{case}.nc"
    ds = xr.open_dataset(f)
    leads = ds["lead"].values.astype(int)
    vy = ds["verify_year"].values.astype(int)
    vm = ds["verify_month"].values.astype(int)
    ds.close()
    out = []
    for ld, y, m in zip(leads, vy, vm):
        if ld < MIN_LEAD:
            continue
        out.append((int(ld), int(y), int(m), f"{int(y):04d}-{int(m):02d}"))
    return sorted(list(set(out)), key=lambda x: x[0])

# ---- plotting ----
for init_key in init_keys:
    cases = groups[init_key]
    if len(cases) == 0:
        continue

    targets = get_verify_targets_from_case(cases[0])
    if len(targets) == 0:
        print("[SKIP] no targets for init:", init_key)
        continue

    print(f"\n[GROUP] init={init_key}  n_cases={len(cases)}  targets={len(targets)}")

    for lead, vy, vm, vkey in targets:
        if (vy not in bw_anom["year"].values) or (vm not in bw_anom["month"].values):
            continue

        ref2d = bw_anom.sel(year=vy, month=vm)

        exps = []
        for case in cases:
            f_anom = OUT_HIND_ANOM / f"{VAR_TAG}_hind_monthly_anom_{case}.nc"
            if not f_anom.exists():
                continue
            ds_h = xr.open_dataset(f_anom)
            hv = f"{VAR_TAG}_hind_anom"
            if hv not in ds_h:
                ds_h.close()
                continue
            z = ds_h[hv]  # (member,lead,lat,lon)
            if lead not in z["lead"].values:
                ds_h.close()
                continue
            ens2d = z.sel(lead=lead).mean("member").load()
            ds_h.close()

            exps.append({
                "case": case,
                "ens": ens2d,
                "acc": lookup_acc(case, vkey, lead),
                "ozone": infer_ozone_mode(case),
                "fam": infer_family_short(case),
            })

        if len(exps) == 0:
            continue

        vlim = compute_common_vlim([ref2d] + [e["ens"] for e in exps], q=VLIM_Q)

        n_maps = 1 + len(exps)
        ncols = MAX_MAPS_PER_ROW
        nrows_maps = int(np.ceil(n_maps / ncols))
        bar_row_h = max(BAR_ROW_BASE, BAR_ROW_BASE + BAR_ROW_PER_BAR * max(0, len(exps) - 4))
        height_ratios = [1.0]*nrows_maps + [bar_row_h]

        fig = plt.figure(figsize=FIGSIZE)
        gs = fig.add_gridspec(
            nrows=nrows_maps + 1, ncols=ncols,
            height_ratios=height_ratios,
            hspace=0.18, wspace=0.06
        )

        # maps
        im_last = None
        idx = 0
        for k in range(n_maps):
            r = idx // ncols
            c = idx % ncols
            ax = fig.add_subplot(gs[r, c], projection=MAP_PROJ)

            if k == 0:
                ttl = f"REF (BWCN) anom — verify {vkey} (lead={lead})"
                im_last = plot_polar_map(ax, ref2d, ttl, vlim)
            else:
                e = exps[k-1]
                ttl = f"{e['case']} ({e['fam']}, {e['ozone']})"
                im_last = plot_polar_map(ax, e["ens"], ttl, vlim)

            idx += 1

        # hide empty slots
        total_slots = nrows_maps * ncols
        for j in range(n_maps, total_slots):
            r = j // ncols
            c = j % ncols
            ax_empty = fig.add_subplot(gs[r, c])
            ax_empty.axis("off")

        # right colorbar
        cax = fig.add_axes([0.92, 0.28, 0.015, 0.55])
        cb = fig.colorbar(im_last, cax=cax, orientation="vertical")
        cb.ax.tick_params(labelsize=FS_TICK)
        cb.set_label(f"{VAR_TAG} anomaly (m)", fontsize=FS_LABEL)

        # ACC bar
        axb = fig.add_subplot(gs[-1, :])
        labels = [e["case"] for e in exps]
        accs = np.array([e["acc"] for e in exps], dtype=float)
        y = np.arange(len(labels), dtype=float) * BAR_Y_SPACING

        axb.barh(y, accs, height=BAR_HEIGHT)
        axb.set_yticks(y)
        axb.set_yticklabels(labels, fontsize=FS_TICK)
        axb.invert_yaxis()
        axb.axvline(0.0, linewidth=0.8, alpha=0.8)
        axb.set_xlim(-1.0, 1.0)
        axb.set_xlabel("ACC (ensemble mean)", fontsize=FS_LABEL)
        axb.set_title(f"ACC vs REF — init {init_key} | verify {vkey} | lead={lead}", fontsize=FS_TITLE, pad=4)
        axb.tick_params(axis="x", labelsize=FS_TICK)

        if len(y) > 0:
            axb.set_ylim(y.max() + BAR_Y_SPACING*1.0, -BAR_Y_SPACING*1.0)

        for i, v in enumerate(accs):
            if np.isfinite(v):
                x = v + (0.03 if v >= 0 else -0.03)
                ha = "left" if v >= 0 else "right"
                axb.text(x, y[i], f"{v:.3f}", va="center", ha=ha, fontsize=FS_TICK)

        fig.suptitle(f"{VAR_TAG} monthly anomalies — init {init_key} → verify {vkey}", fontsize=11, y=0.985)

        out_png = OUT_FIG_DIR / f"monthly_{VAR_TAG}_init{init_key}_verify{vkey}_lead{lead:02d}.png"
        out_pdf = OUT_FIG_DIR / f"monthly_{VAR_TAG}_init{init_key}_verify{vkey}_lead{lead:02d}.pdf"
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        fig.savefig(out_pdf, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print("  [SAVE]", out_png)

print("\n[DONE] figs ->", OUT_FIG_DIR)
